#### 1. Основные функции в координации e-commerce

Аналитик обеспечивает связность процессов, предоставляя единую версию правды (Single Source of Truth) для всех отделов:

*   **Мониторинг KPI в реальном времени:** Аналитик создает и поддерживает дашборды (например, в Apache Superset), отслеживая ключевые показатели: выручка, конверсия (CR), 
средний чек (AOV), стоимость привлечения клиента (CAC), пожизненная ценность (LTV).

*   **Координация маркетинга и продаж:** Аналитик выявляет, какие каналы приносят самую высокую прибыль, и рекомендуют перераспределять бюджет. Он анализирует эффективность акций и промокодов, координируя действия отдела продаж с маркетингом.

*   **Оптимизация воронки продаж (CRO):** Аналитик находит узкие места в пути пользователя (где происходит отказ от корзины, где долгая загрузка) и координирует работу с дизайнерами и разработчиками для улучшения UX/UI.

*   **Синхронизация с логистикой и складом:** Аналитик прогнозирует спрос на основе исторических данных, помогая избежать дефицита (out-of-stock) или затоваривания.

#### 2. Практические сценарии (Co-working)

**Маркетинг** Анализ ROAS (возврат расходов на рекламу), когортный анализ, эффективность e-mail рассылок.

**Product/UX** A/B тестирование новых фич сайта, анализ поведения пользователей (тепловые карты, пути).

**Категорийный менеджмент** ABC/XYZ-анализ ассортимента, поиск товаров с высокой маржой, анализ ценообразования конкурентов.

**Логистика** Анализ скорости доставки, точности комплектации заказов (order accuracy).

#### 3. Ценность для бизнеса

Аналитик данных в роли координатора позволяет:

*   **Принимать обоснованные решения:** Переход от интуитивного управления к управлению на основе данных.

*   **Быстро реагировать:** Мгновенно  замечать аномалии (например, резкое падение конверсии) и координировать их устранение.

*   **Персонализировать опыт:** Сегментировать клиентов для повышения повторных продаж.

В итоге, аналитик данных становится **"мозговым центром"**, который координирует все части e-commerce механизма для максимизации прибыли.

### Работа аналитика в интернет-магазине электронной техники

Жизненный цикл гаджетов короткий, а цена ошибки (затоваривание склада устаревшими моделями или дефицит новинок) крайне высока. 

#### 1. Сбор и подготовка данных (PostgreSQL)

Первым шагом аналитик извлекает исторические данные о продажах, остатках и поступлениях. В электронике важно учитывать не только количество, но и категории (напримеп, "Смартфоны" продаются быстрее, чем "Холодильники").
**Пример  SQL-запроса для выгрузки агрегатных данных:**

In [ ]:
# Настройка путей и импортов
import sys
from pathlib import Path

# Найдём корень проекта (где лежит README.md или config.py)
def find_project_root(marker="config.py"):
    current = Path().resolve()
    while current != current.parent:
        if (current / marker).exists():
            return current
        current = current.parent
    raise RuntimeError(f"Project root with '{marker}' not found!")

PROJECT_ROOT = find_project_root()
print(f"Project root: {PROJECT_ROOT}")

# Добавим корень проекта в sys.path, чтобы импортировать src и config
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

# Путь к данным
CSV_PATH = PROJECT_ROOT / "data" / "raw" / "products-with-falling-demand.csv"
assert CSV_PATH.exists(), f"CSV file not found at {CSV_PATH}"
print(f"CSV file: {CSV_PATH}")

In [ ]:
from src.db.export import export_falling_demand_to_csv
import pandas as pd

# Выгружаем данные
csv_path = export_falling_demand_to_csv()



In [5]:
# Загружаем для анализа
df = pd.read_csv(csv_path)

# Быстрая проверка
df.info()
df.describe()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 659 entries, 0 to 658
Data columns (total 7 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   product_id     659 non-null    int64  
 1   category       659 non-null    object 
 2   model_name     659 non-null    object 
 3   sale_date      659 non-null    object 
 4   daily_sales    659 non-null    int64  
 5   diff           659 non-null    int64  
 6   moving_avg_7d  659 non-null    float64
dtypes: float64(1), int64(3), object(3)
memory usage: 36.2+ KB


,product_id,daily_sales,diff,moving_avg_7d
count,659.000000,659.000000,659.000000,659.000000
mean,10.499241,7.100152,-8.148710,12.112504
std,5.796282,3.545490,5.813911,3.039023
min,1.000000,1.000000,-36.000000,3.430000
25%,6.000000,4.000000,-11.000000,10.140000
50%,11.000000,7.000000,-7.000000,12.140000
75%,15.000000,10.000000,-4.000000,14.215000
max,20.000000,19.000000,-1.000000,20.710000


In [3]:
# Run query 

from src.db.queries import run_query

query = '''
WITH daily_sales AS (
    SELECT 
        p.product_id,
        s.sale_date::date AS sale_date,
        SUM(s.quantity) AS daily_sales
    FROM sales s
    JOIN products p ON s.product_id = p.product_id
    GROUP BY p.product_id, s.sale_date::date
)
SELECT 
    product_id,
    sale_date,
    daily_sales,
    
    -- 1. Накопительный итог (Running Total)
    SUM(daily_sales) OVER (
        PARTITION BY product_id 
        ORDER BY sale_date
    ) AS running_total,
    
    -- 2. Продажи за предыдущий день (Prev Day)
    LAG(daily_sales, 1) OVER (
        PARTITION BY product_id 
        ORDER BY sale_date
    ) AS prev_day_sales,
    
    -- 3. Разница к предыдущему дню (Day-over-Day)
    daily_sales - LAG(daily_sales, 1) OVER (
        PARTITION BY product_id 
        ORDER BY sale_date
    ) AS diff,
    
    -- 4. Порядковый номер продажи (Row Number)
    ROW_NUMBER() OVER (
        PARTITION BY product_id 
        ORDER BY sale_date
    ) AS sale_sequence,
    
    -- 5. Скользящее среднее за 7 дней (Moving Average) 
    ROUND(AVG(daily_sales) OVER (
        PARTITION BY product_id 
        ORDER BY sale_date
        ROWS BETWEEN 6 PRECEDING AND CURRENT ROW
    ), 2) AS moving_avg_7d
    
FROM daily_sales
WHERE product_id IN (1, 2)  -- Для наглядности берём 2 товара
ORDER BY product_id, sale_date
LIMIT 1000;
'''
df = run_query(query)
df


2026-03-07 20:36:57.805 | INFO     | src.db.queries:run_query:28 - Query returned 1000 rows


,product_id,sale_date,daily_sales,running_total,prev_day_sales,diff,sale_sequence,moving_avg_7d
0,1,2024-03-08,6,6.0,NaN,NaN,1,6.00
1,1,2024-03-09,2,8.0,6.0,-4.0,2,4.00
2,1,2024-03-11,1,9.0,2.0,-1.0,3,3.00
3,1,2024-03-12,25,34.0,1.0,24.0,4,8.50
4,1,2024-03-13,5,39.0,25.0,-20.0,5,7.80
...,...,...,...,...,...,...,...,...
995,2,2025-03-31,6,4934.0,12.0,-6.0,384,8.71
996,2,2025-04-01,4,4938.0,6.0,-2.0,385,8.00
997,2,2025-04-02,16,4954.0,4.0,12.0,386,9.14
998,2,2025-04-03,17,4971.0,16.0,1.0,387,10.43


In [4]:
# Run query 

from src.db.queries import run_query

query = '''
-- 1. FROM/JOIN (Собираем данные из таблиц)
-- 2. WHERE (Фильтруем строки)
-- 3. GROUP BY (Группируем строки)
-- 4. SELECT (Вычисляем агрегаты и выдаем результат)


-- ОТЧЁТ: Поиск товаров с признаками падения спроса
-- ЦЕЛЬ: Выявить товары, где продажи снизились И находятся ниже средней нормы
-- ИСПОЛЬЗОВАНИЕ: Ежедневный мониторинг ассортимента для отдела закупок/продаж


WITH daily_sales AS (

    -- ШАГ 1: Агрегация продаж по дням
    -- Превращаем транзакции (время) в дневные итоги (дата)

    SELECT 
        p.product_id,
        p.model_name,
        p.category,
        s.sale_date::date AS sale_date,           -- Приводим к дате (убираем время)
        SUM(s.quantity) AS daily_sales            -- Сумма продаж за день
    FROM sales s
    JOIN products p ON s.product_id = p.product_id
    WHERE s.sale_date >= CURRENT_DATE - INTERVAL '90 days'  -- Последние 90 дней
    GROUP BY p.product_id, p.model_name, p.category, s.sale_date::date
),

sales_metrics AS (

    -- ШАГ 2: Расчёт аналитических метрик
    -- Используем оконные функции для сравнения с прошлым и средней нормой
	
    SELECT 
        product_id,
        model_name,
        category,
        sale_date,
        daily_sales,
        
        -- Разница к предыдущему дню продаж (Day-over-Day)
        -- Если NULL (первый день), ставим 0 для чистоты отчёта
		
        COALESCE(
            daily_sales - LAG(daily_sales) OVER (
                PARTITION BY product_id 
                ORDER BY sale_date
            ), 
            0
        ) AS diff,
        
        -- Скользящее среднее за 7 дней (сглаживает случайные пики)
        -- Помогает увидеть реальный тренд, а не разовые колебания
		
        ROUND(
            AVG(daily_sales) OVER (
                PARTITION BY product_id 
                ORDER BY sale_date 
                ROWS BETWEEN 6 PRECEDING AND CURRENT ROW  -- 6 прошлых + текущий = 7 дней
            ), 
            2
        ) AS moving_avg_7d
        
    FROM daily_sales
)

-- ШАГ 3: Фильтрация «проблемных» товаров
-- Критерии: продажи упали к вчера И ниже средней нормы за неделю

SELECT 
    product_id,
    category,
    model_name,
    sale_date,
    daily_sales AS "Продажи_шт",
    diff AS "Разница_к_вчера",
    moving_avg_7d AS "Среднее_7дней"
FROM sales_metrics
WHERE diff < 0                      -- Продажи снизились к предыдущему дню
  AND daily_sales < moving_avg_7d   -- Продажи ниже средней нормы
ORDER BY sale_date DESC, diff ASC   -- Сначала свежие даты, сильнее падения
LIMIT 100;                          -- Ограничиваем вывод для удобства
'''
df = run_query(query)
df


2026-03-07 20:37:07.526 | INFO     | src.db.queries:run_query:28 - Query returned 100 rows


,product_id,category,model_name,sale_date,Продажи_шт,Разница_к_вчера,Среднее_7дней
0,5,Clothing,Model_5,2026-03-07,3,-4,11.00
1,18,Electronics,Model_18,2026-03-06,4,-12,15.29
2,19,Clothing,Model_19,2026-03-06,12,-10,16.71
3,11,Clothing,Model_11,2026-03-06,8,-6,10.86
4,16,Electronics,Model_16,2026-03-06,8,-3,14.71
...,...,...,...,...,...,...,...
95,6,Electronics,Model_6,2026-02-22,3,-10,16.14
96,16,Electronics,Model_16,2026-02-22,5,-9,8.00
97,4,Electronics,Model_4,2026-02-22,5,-6,11.00
98,5,Clothing,Model_5,2026-02-22,16,-6,19.00
